# RiskFlow PayGuard — IEEE-CIS Fraud-Risk EDA

## Phase 1 objective

This notebook examines the IEEE-CIS Fraud Detection dataset from the perspective
of a production fraud-risk product.

The analysis focuses on:

- source-table contracts
- target imbalance
- identity-data coverage
- missingness and cardinality
- transaction amount behavior
- chronological fraud patterns
- fraud-rate segmentation
- baseline feature selection

Raw Kaggle files remain local and are not committed to Git.


## 1. Setup

The transaction table is the primary table. Identity data is optional and is
linked through `TransactionID`.

The complete transaction and identity tables are loaded separately in this
section. A full wide join is intentionally avoided until it is analytically
necessary.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

from src.data_processing import (
    AMOUNT_COLUMN,
    JOIN_KEY,
    TARGET_COLUMN,
    TIME_COLUMN,
    load_train_tables,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Raw data:     {DATA_DIR.relative_to(PROJECT_ROOT)}")
print(f"Processed:    {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")

Project root: riskflow-payguard
Raw data:     data/raw
Processed:    data/processed


## 2. Load the training source tables

Set `NROWS` to an integer such as `100_000` for quick development.

Use `None` for the complete Phase 1 analysis.


In [2]:
NROWS: int | None = None

transaction, identity = load_train_tables(
    DATA_DIR,
    nrows=NROWS,
)

print(f"Transaction shape: {transaction.shape}")
print(f"Identity shape:    {identity.shape}")

Transaction shape: (590540, 394)
Identity shape:    (144233, 41)


## 3. Dataset dimensions and join coverage

Identity attributes exist for only a subset of transactions. Their absence may
be operationally meaningful, so identity availability should be measured before
deciding how missing identity values will be handled.


In [3]:
transaction_memory_mb = (
    transaction.memory_usage(deep=True).sum() / 1024**2
)
identity_memory_mb = (
    identity.memory_usage(deep=True).sum() / 1024**2
)

identity_key_set = set(identity[JOIN_KEY])
has_identity = transaction[JOIN_KEY].isin(identity_key_set)

identity_coverage = has_identity.mean()
orphan_identity_count = int(
    (~identity[JOIN_KEY].isin(transaction[JOIN_KEY])).sum()
)

dataset_dimensions = pd.DataFrame(
    [
        {
            "table": "train_transaction",
            "rows": len(transaction),
            "columns": transaction.shape[1],
            "unique_transaction_ids": transaction[JOIN_KEY].nunique(),
            "memory_mb": transaction_memory_mb,
        },
        {
            "table": "train_identity",
            "rows": len(identity),
            "columns": identity.shape[1],
            "unique_transaction_ids": identity[JOIN_KEY].nunique(),
            "memory_mb": identity_memory_mb,
        },
    ]
)

display(dataset_dimensions)

print(f"Identity coverage:       {identity_coverage:.2%}")
print(f"Transactions with ID:    {has_identity.sum():,}")
print(f"Transactions without ID: {(~has_identity).sum():,}")
print(f"Orphan identity rows:    {orphan_identity_count:,}")

,table,rows,columns,unique_transaction_ids,memory_mb
0,train_transaction,590540,394,590540,"1,791.6674"
1,train_identity,144233,41,144233,56.5075


Identity coverage:       24.42%
Transactions with ID:    144,233
Transactions without ID: 446,307
Orphan identity rows:    0


In [4]:
assert transaction[JOIN_KEY].is_unique
assert identity[JOIN_KEY].is_unique
assert orphan_identity_count == 0
assert TARGET_COLUMN in transaction.columns
assert transaction[TARGET_COLUMN].notna().all()
assert set(transaction[TARGET_COLUMN].unique()).issubset({0, 1})

print("Source-table contract checks passed.")

Source-table contract checks passed.


## 4. Target imbalance

Fraud detection is an imbalanced classification problem. Accuracy would provide
a misleading view of model quality because legitimate transactions dominate the
dataset.

The initial model evaluation will therefore emphasize ranking, probability
quality, and fraud-capture metrics rather than accuracy.


In [5]:
target_counts = (
    transaction[TARGET_COLUMN]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

legitimate_count = int(target_counts.loc[0])
fraud_count = int(target_counts.loc[1])
total_count = int(target_counts.sum())

fraud_rate = fraud_count / total_count
legitimate_to_fraud_ratio = (
    legitimate_count / fraud_count if fraud_count else float("inf")
)

target_summary = pd.DataFrame(
    [
        {
            "class": "legitimate",
            "target_value": 0,
            "transaction_count": legitimate_count,
            "share": legitimate_count / total_count,
        },
        {
            "class": "fraud",
            "target_value": 1,
            "transaction_count": fraud_count,
            "share": fraud_rate,
        },
    ]
)

display(target_summary)

print(f"Total transactions:          {total_count:,}")
print(f"Fraudulent transactions:     {fraud_count:,}")
print(f"Overall fraud rate:          {fraud_rate:.4%}")
print(
    "Legitimate-to-fraud ratio: "
    f"{legitimate_to_fraud_ratio:,.1f}:1"
)

,class,target_value,transaction_count,share
0,legitimate,0,569877,0.9650
1,fraud,1,20663,0.0350


Total transactions:          590,540
Fraudulent transactions:     20,663
Overall fraud rate:          3.4990%
Legitimate-to-fraud ratio: 27.6:1


### Initial product implication

The observed imbalance will inform:

- `scale_pos_weight` or equivalent class weighting
- PR AUC and ROC AUC reporting
- precision and recall at decision thresholds
- manual-review capacity analysis
- fraud-capture and false-positive trade-offs

Resampling methods such as SMOTE are deferred until the chronological baseline
has been evaluated.
